# NutriVision · Notebook 03 — Asistente nutricional generativo (RAG + LLM local)
**Grupo 8 · Tema 8.** Demostración del asistente que acompaña al clasificador.

**Arquitectura (Retrieval-Augmented Generation ligera):**
1. *Recuperación*: se identifican en la pregunta los alimentos mencionados (coincidencia exacta y difusa sobre los
   nombres en español e inglés) y se suma el último plato detectado por la cámara y el diario calórico del día.
2. *Aumento*: las fichas nutricionales recuperadas se insertan en el *prompt* de sistema con reglas de seguridad.
3. *Generación*: un LLM *decoder-only* local (Qwen2.5-1.5B-Instruct, Transformer de 28 capas) redacta la respuesta.

Así el LLM no inventa cifras: las calorías vienen de la base de datos y el modelo solo las explica.
Ejecutar en Colab con GPU (también funciona en CPU, más lento). Sube la carpeta del proyecto o clónala.

In [6]:
!pip -q install transformers accelerate
import sys, os, zipfile
from google.colab import files
RAIZ = "/content/proyecto_nutricion_ia"          # la carpeta, no el .zip
if not os.path.exists(f"{RAIZ}/src/asistente.py"):
    if os.path.exists("/content/proyecto_nutricion_ia.zip"):   # ya lo subiste: solo se descomprime
        zipfile.ZipFile("/content/proyecto_nutricion_ia.zip").extractall("/content")
    else:
        print("Selecciona el archivo proyecto_nutricion_ia.zip")
        zipfile.ZipFile(list(files.upload().keys())[0]).extractall("/content")
sys.path.insert(0, RAIZ); os.chdir(RAIZ)
from src.nutricion import BaseNutricional, DiarioCalorico, meta_calorica
from src.asistente import AsistenteNutricional
base = BaseNutricional()
diario = DiarioCalorico(base, meta_kcal=meta_calorica("femenino", 21, 60, 165, "ligera"))
bot = AsistenteNutricional(base, diario, motor="transformers")
print(f"Meta: {diario.meta_kcal:.0f} kcal")

Meta: 1877 kcal


## 1. Simulamos detecciones de la cámara y el registro del día

In [7]:
diario.registrar("pancakes")
diario.registrar("caesar_salad")
diario.registrar("pizza", factor=1.5)
bot.ultimo_detectado = "pizza"
print(diario.resumen())

Registro de hoy:
- 00:18: Pancakes x1 = 350 kcal
- 00:18: Ensalada César x1 = 360 kcal
- 00:18: Pizza x1.5 = 848 kcal
Total: 1558 kcal (P 56 g, C 167 g, G 74 g). Meta diaria: 1877 kcal; restante: 320 kcal.


## 2. Contexto que recupera el sistema para una pregunta

In [8]:
contexto, clases = bot.construir_contexto("¿Qué podría comer en vez de la pizza que sea más ligero?")
print(clases); print(contexto)

['pizza']
Último plato detectado por la cámara: Pizza.
Dato: Pizza (2 porciones, ~210 g): 565 kcal, proteína 25 g, carbohidratos 66 g, grasa 22 g. Categoría: plato fuerte.
  Alternativas más ligeras de la misma categoría: Vieiras (200 kcal), Mejillones al vapor (335 kcal), Sushi (rollo) (345 kcal)
Registro de hoy:
- 00:18: Pancakes x1 = 350 kcal
- 00:18: Ensalada César x1 = 360 kcal
- 00:18: Pizza x1.5 = 848 kcal
Total: 1558 kcal (P 56 g, C 167 g, G 74 g). Meta diaria: 1877 kcal; restante: 320 kcal.


## 3. Conversación con el LLM

In [9]:
preguntas = [
    "¿Cuántas calorías tiene lo que acabo de fotografiar?",
    "¿Cuántas calorías llevo hoy y cuánto me queda para la meta?",
    "¿Qué podría comer en vez de la pizza que sea más ligero?",
    "¿El ceviche tiene buena cantidad de proteína?",
    "Tengo diabetes, ¿puedo comer cheesecake?",
]
for p in preguntas:
    print("👤", p); print("🤖", bot.responder(p)); print("-" * 80)

👤 ¿Cuántas calorías tiene lo que acabo de fotografiar?


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

... (2 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

🤖 Lo que has fotografiado es una pizza con dos porciones, cuya calificación es de 565 kilocalorías.
--------------------------------------------------------------------------------
👤 ¿Cuántas calorías llevo hoy y cuánto me queda para la meta?
🤖 Hasta ahora llevaste 1558 kilocalorías. Para cumplir tu meta diaria de 1877 kilocalorías, te falta 320 kilocalorías.
--------------------------------------------------------------------------------
👤 ¿Qué podría comer en vez de la pizza que sea más ligero?
🤖 Podrías probar las siguientes opciones más ligeras:

1. **Vieiras**: Con 200 kilocalorías, son una buena opción vegetariana o vegana si prefieres evitar productos lácteos.
   
2. **Mejillones al vapor**: Con 335 kilocalorías, también puedes considerar como alternativa.

3. **Sushi (rollo)**: Con 345 kilocalorías, es otro plato ligero ideal para el almuerzo o cena.

Recuerda siempre elegir alimentos frescos y variados para mantener un equilibrio nutricional adecuado.
-------------------------

## 4. Comparación con el motor por reglas (sin LLM)
Útil para el informe: el LLM redacta de forma natural y contextual; las reglas solo devuelven plantillas.

In [10]:
reglas = AsistenteNutricional(base, diario, motor="reglas"); reglas.ultimo_detectado = "pizza"
for p in preguntas[:3]:
    print("👤", p); print("📋", reglas.responder(p)); print("-" * 80)

👤 ¿Cuántas calorías tiene lo que acabo de fotografiar?
📋 Pizza (2 porciones, ~210 g): 565 kcal, proteína 25 g, carbohidratos 66 g, grasa 22 g. Categoría: plato fuerte.
(Valores aproximados; varían según porción y preparación.)
--------------------------------------------------------------------------------
👤 ¿Cuántas calorías llevo hoy y cuánto me queda para la meta?
📋 Registro de hoy:
- 00:18: Pancakes x1 = 350 kcal
- 00:18: Ensalada César x1 = 360 kcal
- 00:18: Pizza x1.5 = 848 kcal
Total: 1558 kcal (P 56 g, C 167 g, G 74 g). Meta diaria: 1877 kcal; restante: 320 kcal.
--------------------------------------------------------------------------------
👤 ¿Qué podría comer en vez de la pizza que sea más ligero?
📋 Pizza (2 porciones, ~210 g): 565 kcal, proteína 25 g, carbohidratos 66 g, grasa 22 g. Categoría: plato fuerte.
Alternativas más ligeras: Vieiras (200 kcal), Mejillones al vapor (335 kcal), Sushi (rollo) (345 kcal)
(Valores aproximados; varían según porción y preparación.)
-------